In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/luongdiem/course-llm-embeddings/course_dup_groups.parquet
/kaggle/input/datasets/luongdiem/course-llm-embeddings/embeddings_course_index.json
/kaggle/input/datasets/luongdiem/course-llm-embeddings/llm_profile_embeddings.npy
/kaggle/input/datasets/luongdiem/course-llm-embeddings/llm_profile_no_prereq_followup_embeddings.npy
/kaggle/input/datasets/luongdiem/course-llm-embeddings/raw_course_text_embeddings.npy
/kaggle/input/datasets/luongdiem/course-llm-embeddings/raw_embeddings_course_index.json
/kaggle/input/datasets/luongdiem/cold-scenario/cold_items_test.json
/kaggle/input/datasets/luongdiem/cold-scenario/cold_items_val.json
/kaggle/input/datasets/luongdiem/cold-scenario/test_cold.parquet
/kaggle/input/datasets/luongdiem/cold-scenario/train_cold.parquet
/kaggle/input/datasets/luongdiem/cold-scenario/val_cold.parquet
/kaggle/input/datasets/luongdiem/evaluator/evaluator.py
/kaggle/input/datasets/luongdiem/lightgcn-backbone-cold/g_i.npy
/kaggle/input/datasets/luong

In [2]:
%%writefile coldrec_adapter.py
import copy
import hashlib
import json
import os
import pickle
import sys
import time
import numpy as np
import pandas as pd
import torch

ITEM_IDX_PATH = "/kaggle/input/datasets/luongdiem/lightgcn-backbone-cold/item_index.json"
USER_IDX_PATH = "/kaggle/input/datasets/luongdiem/lightgcn-backbone-cold/user_index_cold.json"
GU_PATH = "/kaggle/input/datasets/luongdiem/lightgcn-backbone-cold/g_u.npy"
GI_PATH = "/kaggle/input/datasets/luongdiem/lightgcn-backbone-cold/g_i.npy"

RAW_IDX_PATH = "/kaggle/input/datasets/luongdiem/course-llm-embeddings/raw_embeddings_course_index.json"
LLM_IDX_PATH = "/kaggle/input/datasets/luongdiem/course-llm-embeddings/embeddings_course_index.json"
RAW_NPY = "/kaggle/input/datasets/luongdiem/course-llm-embeddings/raw_course_text_embeddings.npy"
LLM_NPY = "/kaggle/input/datasets/luongdiem/course-llm-embeddings/llm_profile_embeddings.npy"
LLM_3D_NPY = "/kaggle/input/datasets/luongdiem/course-llm-embeddings/llm_profile_no_prereq_followup_embeddings.npy"

COLD_VAL_JSON = "/kaggle/input/datasets/luongdiem/cold-scenario/cold_items_val.json"
COLD_TEST_JSON = "/kaggle/input/datasets/luongdiem/cold-scenario/cold_items_test.json"
TRAIN_PARQUET = "/kaggle/input/datasets/luongdiem/cold-scenario/train_cold.parquet"
VAL_PARQUET = "/kaggle/input/datasets/luongdiem/cold-scenario/val_cold.parquet"
TEST_PARQUET = "/kaggle/input/datasets/luongdiem/cold-scenario/test_cold.parquet"

OUTPUT_DIR = "/kaggle/working"


def find_file_in_kaggle(filename, default_path=None):
    """Tự động tìm đường dẫn tệp trong /kaggle/input hoặc /kaggle/working."""
    if default_path and os.path.exists(default_path):
        return default_path
    for base_dir in ("/kaggle/input", "/kaggle/working", "."):
        if not os.path.exists(base_dir):
            continue
        for root, _, files in os.walk(base_dir):
            if filename in files:
                return os.path.join(root, filename)
    raise FileNotFoundError(
        f"Không tìm thấy tệp '{filename}' trong /kaggle/input! Hãy kiểm tra"
        " Dataset đã gắn vào Notebook."
    )


# Đường dẫn tệp phân nhóm trùng lặp P-RQ3 (tự động tìm nếu đặt ở thư mục khác)
DUP_GROUPS_PARQUET = find_file_in_kaggle(
    "course_dup_groups.parquet",
    default_path="/kaggle/input/datasets/luongdiem/course-llm-embeddings/course_dup_groups.parquet",
)

# Tự động tìm và import evaluator.py (hoặc evaluate.py)
try:
    eval_file_path = find_file_in_kaggle("evaluator.py")
    sys.path.insert(0, os.path.dirname(eval_file_path))
    from evaluator import evaluate, topk_random_ties
except FileNotFoundError:
    eval_file_path = find_file_in_kaggle("evaluate.py")
    sys.path.insert(0, os.path.dirname(eval_file_path))
    from evaluate import evaluate, topk_random_ties


def md5_hash(filepath):
    """Tính mã băm MD5 của tệp để truy vết phiên bản"""
    return hashlib.md5(open(filepath, "rb").read()).hexdigest()


def to_bool_mask(ids, c2i, n_items=3769):
    """Chuyển danh sách course_id sang mặt nạ bool dài 3.769"""
    m = torch.zeros(n_items, dtype=torch.bool)
    m[[c2i[str(c)] for c in ids]] = True
    return m


def build_hist(df, u2i, c2i):
    """Dựng từ điển lịch sử tương tác {u_idx: [i_idx, ...]}."""
    u_mapped = df["user_id"].astype(str).map(u2i)
    i_mapped = df["course_id"].astype(str).map(c2i)
    temp = pd.DataFrame({"u": u_mapped, "i": i_mapped}).dropna()
    temp["u"] = temp["u"].astype(int)
    temp["i"] = temp["i"].astype(int)
    return temp.groupby("u")["i"].apply(list).to_dict()


def _extract_id_list(json_path, key_name):
    data = json.load(open(json_path))
    items = data[key_name] if isinstance(data, dict) else data
    return [str(x) for x in items]


# Lớp bộ chuyển đổi dữ liệu và kiểm tra giao thức
class ColdRecDataAdapter:

    def __init__(self, output_dir=OUTPUT_DIR):
        self.output_dir = output_dir
        os.makedirs(self.output_dir, exist_ok=True)
        self._load_and_verify_all()

    def _load_and_verify_all(self):
        print("=== KIỂM TRA MD5 VÀ BẢNG ÁNH XẠ CHỈ SỐ CHUNG ===")
        # Đọc và đối chiếu bảng ánh xạ chỉ số chung 
        self.course_ids = _extract_id_list(ITEM_IDX_PATH, "course_id")
        self.users = _extract_id_list(USER_IDX_PATH, "user_id")

        assert len(self.course_ids) == 3769 and self.course_ids == sorted(self.course_ids), "Lỗi bảng item_index.json!"
        assert len(self.users) == 98220 and self.users == sorted(self.users), "Lỗi bảng user_index_cold.json!"

        if os.path.exists(RAW_IDX_PATH):
            raw_cids = _extract_id_list(RAW_IDX_PATH, "course_id")
            assert self.course_ids == raw_cids, "item_index.json không khớp raw_embeddings_course_index.json!"
        if os.path.exists(LLM_IDX_PATH):
            llm_cids = _extract_id_list(LLM_IDX_PATH, "course_id")
            assert self.course_ids == llm_cids, "item_index.json không khớp embeddings_course_index.json!"

        self.item2idx = {c: i for i, c in enumerate(self.course_ids)}
        self.user2idx = {u: i for i, u in enumerate(self.users)}
        self.n_users = len(self.users)
        self.n_items = len(self.course_ids)

        # Đọc danh sách khóa lạnh (251 cold-val, 502 cold-test)
        self.cold_val_ids = [str(c) for c in json.load(open(COLD_VAL_JSON))]
        self.cold_test_ids = [str(c) for c in json.load(open(COLD_TEST_JSON))]
        assert len(self.cold_val_ids) == 251 and len(self.cold_test_ids) == 502

        self.cold_val_idx = sorted([self.item2idx[c] for c in self.cold_val_ids])
        self.cold_test_idx = sorted([self.item2idx[c] for c in self.cold_test_ids])
        self.cold_idx = np.array(sorted(self.cold_val_idx + self.cold_test_idx), dtype=np.int64)
        assert len(self.cold_idx) == 753

        # Đọc các tệp tương tác và chạy assert chống rò rỉ
        self.tr_cold = pd.read_parquet(TRAIN_PARQUET)
        self.va_cold = pd.read_parquet(VAL_PARQUET)
        self.te_cold = pd.read_parquet(TEST_PARQUET)

        tr_items = set(self.tr_cold["course_id"].astype(str))
        cv, ct = set(self.cold_val_ids), set(self.cold_test_ids)
        assert not (cv & tr_items) and not (ct & tr_items) and not (cv & ct), "Rò rỉ khóa lạnh vào train hoặc trùng val/test!"
        assert set(self.va_cold["course_id"].astype(str)) <= cv
        assert set(self.te_cold["course_id"].astype(str)) <= ct
        assert set(self.va_cold["user_id"].astype(str)) <= set(self.tr_cold["user_id"].astype(str))
        assert set(self.te_cold["user_id"].astype(str)) <= set(self.tr_cold["user_id"].astype(str))

        # Xác định đúng 2.401 khóa ấm có tương tác trong train_cold
        self.warm_active_idx = np.unique(
            self.tr_cold["course_id"]
            .astype(str)
            .map(self.item2idx)
            .to_numpy(dtype=np.int64)
        )
        assert len(self.warm_active_idx) == 2401 and not (set(self.warm_active_idx.tolist()) & set(self.cold_idx.tolist()))

        # Đọc và kiểm tra Backbone LightGCN (g_u, g_i)
        self.g_u = np.load(GU_PATH).astype(np.float32)
        self.g_i = np.load(GI_PATH).astype(np.float32)

        gu_md5, gi_md5 = md5_hash(GU_PATH), md5_hash(GI_PATH)
        print(f"MD5 g_u.npy: {gu_md5} (Kỳ vọng: 7a170558...)")
        print(f"MD5 g_i.npy: {gi_md5} (Kỳ vọng: d17ad9df...)")

        assert self.g_u.shape == (98220, 64) and self.g_i.shape == (3769, 64)
        assert np.allclose(self.g_i[self.cold_idx], 0.0), "Khóa lạnh (753 khóa) phải có g_i = 0!"

        # Đảm bảo 615 khóa ấm không có tương tác trong train_cold cũng có g_i = 0
        inactive_mask = np.ones(self.n_items, dtype=bool)
        inactive_mask[self.warm_active_idx] = False
        assert np.allclose(self.g_i[inactive_mask], 0.0), "615 khóa ấm không tương tác phải có g_i = 0!"

        # Đọc và chuẩn hóa L2 cho các biến thể đặc trưng nội dung (raw, llm, fused, llm_3d, fused_3d)
        print(f"MD5 raw_emb: {md5_hash(RAW_NPY)} | MD5 llm_emb: {md5_hash(LLM_NPY)}")

        nz = lambda X: X / np.linalg.norm(X, axis=1, keepdims=True).clip(1e-9)
        self.S_raw = nz(np.load(RAW_NPY).astype(np.float32))
        self.S_llm = nz(np.load(LLM_NPY).astype(np.float32))
        assert self.S_raw.shape == (3769, 384) and self.S_llm.shape == (3769,384,)
        self.S_fused = np.concatenate([self.S_raw, self.S_llm], axis=1).astype(np.float32)  # [3769, 768]

        self.contents = {
            "raw": self.S_raw,
            "llm": self.S_llm,
            "fused": self.S_fused,
        }

        if os.path.exists(LLM_3D_NPY):
            print(f"MD5 llm_3d_emb: {md5_hash(LLM_3D_NPY)}")
            self.S_llm_3d = nz(np.load(LLM_3D_NPY).astype(np.float32))
            assert self.S_llm_3d.shape == (3769, 384)
            self.S_fused_3d = np.concatenate(
                [self.S_raw, self.S_llm_3d], axis=1
            ).astype(np.float32)
            self.contents["llm_3d"] = self.S_llm_3d
            self.contents["fused_3d"] = self.S_fused_3d

        # Dựng từ điển mặt nạ lịch sử, đích đánh giá và mặt nạ ứng viên (a), (b)
        self.hist_tr = build_hist(self.tr_cold, self.user2idx, self.item2idx)
        self.hist_trva = build_hist(
            pd.concat([self.tr_cold, self.va_cold], ignore_index=True),
            self.user2idx,
            self.item2idx,
        )
        self.val_target_dict = build_hist(
            self.va_cold, self.user2idx, self.item2idx
        )
        self.test_target_dict = build_hist(
            self.te_cold, self.user2idx, self.item2idx
        )
        assert (
            len(self.val_target_dict) == 2092
            and len(self.test_target_dict) == 3340
        )

        self.cand_val_a = to_bool_mask(
            self.cold_val_ids, self.item2idx, self.n_items
        )
        self.cand_test_a = to_bool_mask(
            self.cold_test_ids, self.item2idx, self.n_items
        )
        self.cand_test_b = torch.ones(self.n_items, dtype=torch.bool)

        # Phân nhóm P-RQ3 ("trung" vs "moi") từ course_dup_groups.parquet
        dup_df = pd.read_parquet(DUP_GROUPS_PARQUET)
        c2dup = dict(zip(dup_df["course_id"].astype(str), dup_df["dup_group_id"]))
        active_warm_course_ids = [self.course_ids[i] for i in self.warm_active_idx]
        warm_active_dup_groups = {c2dup[c] for c in active_warm_course_ids}

        self.cold_test_dup_ids = [c for c in self.cold_test_ids if c2dup[c] in warm_active_dup_groups]
        self.cold_test_new_ids = [
            c
            for c in self.cold_test_ids
            if c2dup[c] not in warm_active_dup_groups
        ]
        assert (
            len(self.cold_test_dup_ids) == 124
            and len(self.cold_test_new_ids) == 378
        ), (
            f"Sai số lượng khóa trùng/mới: {len(self.cold_test_dup_ids)} /"
            f" {len(self.cold_test_new_ids)}"
        )

        self.groups_prq3 = {
            "trung": {self.item2idx[c] for c in self.cold_test_dup_ids},
            "moi": {self.item2idx[c] for c in self.cold_test_new_ids},
        }
        print("=== HOÀN TẤT KIỂM TRA: MỌI ASSERT ĐỀU ĐẠT CHUẨN! ===\n")

    def export_coldrec_files(
        self, dataset_name="MOOCCubeX", content_variant="raw"
    ):
        """Xuất bộ dữ liệu sang đúng cấu trúc thư mục mà ColdRec (ALDI, CGRC, USIM)

        và mã chính thức zfnWong/ALDI yêu cầu, giữ nguyên chỉ số 0..98219 và
        0..3768.
        """
        ds_dir = os.path.join(self.output_dir, "ColdRec_Data", dataset_name)
        os.makedirs(ds_dir, exist_ok=True)

        # Ánh xạ các bảng tương tác sang chỉ số nguyên (user, item)
        def to_mapped_df(df):
            u = df["user_id"].astype(str).map(self.user2idx).astype(np.int64)
            i = df["course_id"].astype(str).map(self.item2idx).astype(np.int64)
            return pd.DataFrame({"user": u, "item": i})

        tr_df = to_mapped_df(self.tr_cold)
        va_df = to_mapped_df(self.va_cold)
        te_df = to_mapped_df(self.te_cold)

        # Lưu các tệp CSV tương tác cho ColdRec
        tr_df.to_csv(os.path.join(ds_dir, "warm_train.csv"), index=False)
        va_df.to_csv(os.path.join(ds_dir, "cold_item_val.csv"), index=False)
        te_df.to_csv(os.path.join(ds_dir, "cold_item_test.csv"), index=False)

        # Lưu danh sách ID khóa ấm có tương tác (2.401 khóa) và khóa lạnh
        np.save(os.path.join(ds_dir, "warm_item_ids.npy"), self.warm_active_idx)
        np.save(
            os.path.join(ds_dir, "cold_val_item_ids.npy"),
            np.array(self.cold_val_idx, dtype=np.int64),
        )
        np.save(
            os.path.join(ds_dir, "cold_test_item_ids.npy"),
            np.array(self.cold_test_idx, dtype=np.int64),
        )
        np.save(os.path.join(ds_dir, "cold_item_ids.npy"), self.cold_idx)

        # Lưu Pre-trained Teacher Embeddings (LightGCN nền chung)
        warm_emb_concat = np.concatenate([self.g_u, self.g_i], axis=0)  # [98220 + 3769, 64]
        np.save(
            os.path.join(ds_dir, f"{dataset_name}_lightgcn.npy"),
            warm_emb_concat,
        )
        np.save(os.path.join(ds_dir, "warm_emb.npy"), warm_emb_concat)
        np.save(os.path.join(ds_dir, "g_u.npy"), self.g_u)
        np.save(os.path.join(ds_dir, "g_i.npy"), self.g_i)

        # Lưu ma trận đặc trưng nội dung khóa học theo biến thể được chọn ('raw', 'llm', 'fused')
        content_mat = self.contents[content_variant]
        np.save(
            os.path.join(ds_dir, f"{dataset_name}_item_content.npy"),
            content_mat,
        )
        np.save(os.path.join(ds_dir, "item_content.npy"), content_mat)

        # Lưu từ điển thông tin tổng hợp (para_dict.pkl) cho bộ nạp của ColdRec / ALDI
        para_dict = {
            "user_num": self.n_users,
            "item_num": self.n_items,
            "warm_items": set(self.warm_active_idx.tolist()),
            "cold_val_items": set(self.cold_val_idx),
            "cold_test_items": set(self.cold_test_idx),
            "cold_items": set(self.cold_idx.tolist()),
            "emb_dim": 64,
            "content_dim": int(content_mat.shape[1]),
            "user_his": self.hist_tr,
            "val_user_his": self.hist_trva,
        }
        with open(os.path.join(ds_dir, "para_dict.pkl"), "wb") as f:
            pickle.dump(para_dict, f)

        print(
            f"[OK] Đã xuất dữ liệu ColdRec ({content_variant},"
            f" dim={content_mat.shape[1]}) tại: {ds_dir}"
        )
        return ds_dir


# Các hàm đánh giá của nhóm (tập a, b, P-RQ3, P-RQ4)
@torch.no_grad()
def eval_subgroups(
    score_fn, targets, mask, cand, groups, ks=(5, 10, 20), tie_seed=0, bs=1024
):
    """Hàm tính P-RQ3 đúng chuẩn:

    Xếp hạng trên TOÀN BỘ 502 khóa của tập (a), sau đó chỉ tính chỉ số trên đích
    thuộc từng nhóm.
    """
    users, K = sorted(targets), max(ks)
    g = torch.Generator().manual_seed(tie_seed)
    disc = 1.0 / np.log2(np.arange(2, K + 2))
    cand = torch.as_tensor(cand)
    out = {
        n: {f"{m}@{k}": [] for m in ("Recall", "NDCG", "HR") for k in ks}
        for n in groups
    }
    for s in range(0, len(users), bs):
        ub = users[s : s + bs]
        sc = score_fn(torch.tensor(ub)).float().cpu().clone()
        sc[:, ~cand] = -float("inf")
        for r, u in enumerate(ub):
            if mask.get(u):
                sc[r, list(mask[u])] = -float("inf")
        top = topk_random_ties(sc, K, g)
        for r, u in enumerate(ub):
            for n, gset in groups.items():
                gt = [i for i in targets[u] if i in gset]
                if not gt:
                    continue
                hit = np.isin(top[r], gt).astype(float)
                for k in ks:
                    h = hit[:k]
                    idcg = disc[: min(len(gt), k)].sum()
                    out[n][f"Recall@{k}"].append(h.sum() / len(gt))
                    out[n][f"NDCG@{k}"].append((h * disc[:k]).sum() / idcg)
                    out[n][f"HR@{k}"].append(float(h.sum() > 0))
    return {n: {m: np.array(v) for m, v in d.items()} for n, d in out.items()}


@torch.no_grad()
def evaluate_coldrec_embeddings(
    adapter, u_emb, i_emb, model_name, variant_name, seed, device="cuda:0"
):
    """Hàm cầu nối bắt buộc:

    Nhận u_emb [98220, 64] và i_emb [3769, 64] xuất ra từ mô hình ColdRec (ALDI
    / CGRC / USIM), chấm điểm bằng evaluator.py và lưu đầy đủ file phục
    vụ Wilcoxon + P-RQ3.
    """
    u_t = torch.as_tensor(u_emb, dtype=torch.float32, device=device)
    i_t = torch.as_tensor(i_emb, dtype=torch.float32, device=device)
    assert u_t.shape == (98220, 64) and i_t.shape == (3769, 64)

    score_fn = lambda ub: (u_t[ub.to(device)] @ i_t.t()).cpu()

    # Đánh giá trên tập (a) (502 khóa cold-test) và tập (b) (3.769 khóa)
    kw = dict(mask=adapter.hist_trva, ks=(5, 10, 20), tie_seed=seed)
    res_a = evaluate(score_fn, adapter.test_target_dict, cand=adapter.cand_test_a, **kw)
    res_b = evaluate(score_fn, adapter.test_target_dict, cand=adapter.cand_test_b, **kw)

    # Đánh giá P-RQ3 theo nhóm khóa trùng / mới
    res_prq3 = eval_subgroups(
        score_fn,
        adapter.test_target_dict,
        adapter.hist_trva,
        adapter.cand_test_a,
        adapter.groups_prq3,
        ks=(5, 10, 20),
        tie_seed=seed,
    )
    assert (
        len(res_prq3["trung"]["Recall@20"]) == 1028
        and len(res_prq3["moi"]["Recall@20"]) == 2844
    )

    # Lưu biểu diễn khóa học của seed này 
    rep_dir = os.path.join(adapter.output_dir, f"results_{model_name}", "item_reps")
    os.makedirs(rep_dir, exist_ok=True)
    np.save(os.path.join(rep_dir, f"{model_name}_{variant_name}_seed{seed}.npy"),i_t.cpu().numpy(),)

    return res_a, res_b, res_prq3


if __name__ == "__main__":
    # Khởi tạo Adapter, chạy toàn bộ kiểm tra assert và xuất dữ liệu cho 3 biến thể nội dung
    adapter = ColdRecDataAdapter()
    for var in ["raw", "llm", "fused"]:
        adapter.export_coldrec_files(
            dataset_name=f"MOOCCubeX_{var}", content_variant=var
        )

Writing coldrec_adapter.py


In [3]:
!python coldrec_adapter.py

=== KIỂM TRA MD5 VÀ BẢNG ÁNH XẠ CHỈ SỐ CHUNG ===
MD5 g_u.npy: 7a170558642de73d301970e8ce7f0a86 (Kỳ vọng: 7a170558...)
MD5 g_i.npy: d17ad9dfc4f4d6cb6de915c8649070d7 (Kỳ vọng: d17ad9df...)
MD5 raw_emb: f80445649ef8a8217d048bd3e7ac0ad5 | MD5 llm_emb: fc1819307be227a1544e1be0a4017393
MD5 llm_3d_emb: 1af436095463ad0af3ef1203ec0ebd6c
=== HOÀN TẤT KIỂM TRA: MỌI ASSERT ĐỀU ĐẠT CHUẨN! ===

[OK] Đã xuất dữ liệu ColdRec (raw, dim=384) tại: /kaggle/working/ColdRec_Data/MOOCCubeX_raw
[OK] Đã xuất dữ liệu ColdRec (llm, dim=384) tại: /kaggle/working/ColdRec_Data/MOOCCubeX_llm
[OK] Đã xuất dữ liệu ColdRec (fused, dim=768) tại: /kaggle/working/ColdRec_Data/MOOCCubeX_fused


In [4]:
# Clone ColdRec (chứa ALDI, CGRC, USIM) và repo chính thức để đối chiếu hàm mất mát
!git clone https://github.com/YuanchenBei/ColdRec.git
!git clone https://github.com/zfnWong/ALDI.git
!git clone https://github.com/Ruochen1003/USIM.git
!cd ColdRec && git rev-parse HEAD  # Ghi lại commit hash 

Cloning into 'ColdRec'...
remote: Enumerating objects: 561, done.
remote: Counting objects: 100% (101/101), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 561 (delta 69), reused 67 (delta 51), pack-reused 460 (from 1)
Receiving objects: 100% (561/561), 2.36 MiB | 24.70 MiB/s, done.
Resolving deltas: 100% (359/359), done.
Cloning into 'ALDI'...
remote: Enumerating objects: 51, done.
remote: Counting objects: 100% (51/51), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 51 (delta 12), reused 45 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (51/51), 38.63 MiB | 31.32 MiB/s, done.
Resolving deltas: 100% (12/12), done.
Cloning into 'USIM'...
remote: Enumerating objects: 79, done.
remote: Counting objects: 100% (79/79), done.
remote: Compressing objects: 100% (71/71), done.
remote: Total 79 (delta 20), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (79/79), 38.02 MiB | 11.05 MiB/s, done.
Resolving deltas: 100% (20/20)

In [5]:
%%writefile run_aldi.py
import argparse
import copy
import itertools
import json
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from coldrec_adapter import ColdRecDataAdapter, evaluate_coldrec_embeddings
from evaluator import evaluate


def set_seed(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)


class ALDIStudent(nn.Module):
    """Mạng ánh xạ nội dung khóa học sang không gian nhúng cộng tác của ALDI"""

    def __init__(self, d_in, d_emb=64, h=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, h),
            nn.GELU(),
            nn.Linear(h, d_emb),
        )

    def forward(self, x):
        return self.net(x)


def get_item_embeddings(model, g_i_t, content_t, warm_active_mask):
    """Khóa ấm có tương tác giữ nguyên g_i; khóa lạnh dùng biểu diễn từ mạng học sinh."""
    s_proj = model(content_t)
    i_emb = torch.where(warm_active_mask.unsqueeze(1), g_i_t, s_proj)
    return i_emb, s_proj


@torch.no_grad()
def val_ndcg20(model, adapter, g_u_t, g_i_t, content_t, warm_active_mask, seed, device):
    model.eval()
    _, s_proj = get_item_embeddings(model, g_i_t, content_t, warm_active_mask)
    # Trên cold-val (tập a), mọi ứng viên đều là khóa lạnh nên chấm bằng s_proj
    score_fn = lambda ub: (g_u_t[ub.to(device)] @ s_proj.t()).cpu()
    res = evaluate(
        score_fn,
        adapter.val_target_dict,
        mask=adapter.hist_tr,
        cand=adapter.cand_val_a,
        ks=(20,),
        tie_seed=seed,
    )
    return float(np.mean(res["NDCG@20"]))


def train_aldi_one_run(
    adapter, g_u_t, g_i_t, content_t, train_u_t, train_i_t, warm_active_t, warm_active_mask, cfg, seed, device,
    max_epochs=150, eval_every=5, patience=6, batch_size=4096
):
    set_seed(seed)
    model = ALDIStudent(d_in=content_t.shape[1], d_emb=64, h=256).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["wd"])

    best_val, best_state, bad, best_epoch = -1.0, None, 0, 0
    n_edges = len(train_u_t)

    for epoch in range(1, max_epochs + 1):
        model.train()
        perm = torch.randperm(n_edges, device=device)
        for start in range(0, n_edges, batch_size):
            idx = perm[start : start + batch_size]
            u_b = train_u_t[idx]
            pos_b = train_i_t[idx]
            neg_b = warm_active_t[torch.randint(len(warm_active_t), (len(idx),), device=device)]

            opt.zero_grad()
            s_pos = model(content_t[pos_b])
            s_neg = model(content_t[neg_b])

            u_vec = g_u_t[u_b]
            t_pos = g_i_t[pos_b]
            t_neg = g_i_t[neg_b]

            # Ranking loss (Student BPR trên khóa ấm có tương tác)
            stu_diff = (u_vec * (s_pos - s_neg)).sum(dim=-1)
            loss_rank = -F.logsigmoid(stu_diff).mean()

            # Rating distribution alignment (KL / Logit matching với Teacher LightGCN)
            tea_diff = (u_vec * (t_pos - t_neg)).sum(dim=-1).detach()
            loss_dist = F.binary_cross_entropy_with_logits(
                stu_diff / cfg["tau"], torch.sigmoid(tea_diff / cfg["tau"])
            )

            # Embedding alignment giữa Student và Teacher trên khóa ấm
            w_uniq = torch.unique(torch.cat([pos_b, neg_b]))
            loss_emb = F.mse_loss(model(content_t[w_uniq]), g_i_t[w_uniq].detach())

            loss = loss_rank + cfg["alpha"] * loss_dist + cfg["beta"] * loss_emb
            loss.backward()
            opt.step()

        if epoch % eval_every == 0:
            v = val_ndcg20(model, adapter, g_u_t, g_i_t, content_t, warm_active_mask, seed, device)
            if v > best_val:
                best_val = v
                best_state = copy.deepcopy(model.state_dict())
                bad = 0
                best_epoch = epoch
            else:
                bad += 1
                if bad >= patience:
                    break

    model.load_state_dict(best_state)
    return model, best_val, best_epoch


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--variant", type=str, required=True, choices=["raw", "llm", "fused"])
    parser.add_argument("--device", type=str, default="cuda:0")
    args = parser.parse_args()

    device = args.device if torch.cuda.is_available() else "cpu"
    adapter = ColdRecDataAdapter()
    out_dir = os.path.join(adapter.output_dir, "results_aldi", args.variant)
    os.makedirs(out_dir, exist_ok=True)

    g_u_t = torch.tensor(adapter.g_u, device=device)
    g_i_t = torch.tensor(adapter.g_i, device=device)
    content_t = torch.tensor(adapter.contents[args.variant], device=device)

    train_u_t = torch.tensor(
        adapter.tr_cold["user_id"].astype(str).map(adapter.user2idx).to_numpy(dtype=np.int64),
        device=device,
    )
    train_i_t = torch.tensor(
        adapter.tr_cold["course_id"].astype(str).map(adapter.item2idx).to_numpy(dtype=np.int64),
        device=device,
    )
    warm_active_t = torch.tensor(adapter.warm_active_idx, device=device)
    warm_active_mask = torch.zeros(adapter.n_items, dtype=torch.bool, device=device)
    warm_active_mask[warm_active_t] = True

    # Lưới siêu tham số: lấy mẫu đúng 20 cấu hình với random.Random(42)
    space = dict(
        lr=[1e-4, 5e-4, 1e-3, 5e-3, 1e-2],
        alpha=[0.1, 0.5, 1.0],
        beta=[0.1, 0.5, 1.0],
        tau=[0.5, 1.0],
        wd=[1e-5, 1e-4, 1e-3],
    )
    grid = [dict(zip(space, v)) for v in itertools.product(*space.values())]
    trials = random.Random(42).sample(grid, 20)

    print(f"=== [ALDI - {args.variant}] BẮT ĐẦU TINH CHỈNH 20 CẤU HÌNH (SEED 42) ===")
    best_cfg, best_score = None, -1.0
    for idx, cfg in enumerate(trials, 1):
        _, val_sc, ep = train_aldi_one_run(
            adapter, g_u_t, g_i_t, content_t, train_u_t, train_i_t,
            warm_active_t, warm_active_mask, cfg, seed=42, device=device
        )
        print(f"Trial {idx:02d}/20 | cfg={cfg} | Best Val NDCG@20={val_sc:.4f} (ep {ep})")
        if val_sc > best_score:
            best_score = val_sc
            best_cfg = cfg

    with open(os.path.join(out_dir, f"best_cfg_{args.variant}.json"), "w") as f:
        json.dump({"best_cfg": best_cfg, "best_val_ndcg20": best_score}, f, indent=2)
    print(f"==> Cấu hình tốt nhất ({args.variant}): {best_cfg} (Val NDCG@20 = {best_score:.4f})")

    # Chạy 5 seed độc lập với cấu hình tốt nhất
    SEEDS = [42, 2023, 2024, 1007, 999]
    rows_a, rows_b, rows_prq3 = [], [], []
    user_ndcg20_seeds = []

    for s in SEEDS:
        model, val_sc, ep = train_aldi_one_run(
            adapter, g_u_t, g_i_t, content_t, train_u_t, train_i_t,
            warm_active_t, warm_active_mask, best_cfg, seed=s, device=device
        )
        model.eval()
        with torch.no_grad():
            i_emb, _ = get_item_embeddings(model, g_i_t, content_t, warm_active_mask)
            res_a, res_b, res_prq3 = evaluate_coldrec_embeddings(
                adapter, g_u_t, i_emb, model_name="aldi", variant_name=args.variant, seed=s, device=device
            )

        user_ndcg20_seeds.append(res_a["NDCG@20"])
        rows_a.append({"seed": s, "best_epoch": ep, **{k: float(np.mean(v)) for k, v in res_a.items()}})
        rows_b.append({"seed": s, "best_epoch": ep, **{k: float(np.mean(v)) for k, v in res_b.items()}})
        rows_prq3.append({
            "seed": s,
            "trung_R@20": float(np.mean(res_prq3["trung"]["Recall@20"])),
            "trung_NDCG@20": float(np.mean(res_prq3["trung"]["NDCG@20"])),
            "moi_R@20": float(np.mean(res_prq3["moi"]["Recall@20"])),
            "moi_NDCG@20": float(np.mean(res_prq3["moi"]["NDCG@20"])),
        })
        print(
            f"[Seed {s}] Test (a) R@20={rows_a[-1]['Recall@20']:.4f}, NDCG@20={rows_a[-1]['NDCG@20']:.4f} | "
            f"P-RQ3 R@20 trùng/mới = {rows_prq3[-1]['trung_R@20']:.4f} / {rows_prq3[-1]['moi_R@20']:.4f}"
        )

    # Lưu CSV kết quả và tệp user_ndcg20_aldi_{variant}.npy (trung bình 5 seed cho 3.340 học viên)
    pd.DataFrame(rows_a).to_csv(os.path.join(out_dir, f"aldi_{args.variant}_5seeds_cand_a.csv"), index=False)
    pd.DataFrame(rows_b).to_csv(os.path.join(out_dir, f"aldi_{args.variant}_5seeds_cand_b.csv"), index=False)
    pd.DataFrame(rows_prq3).to_csv(os.path.join(out_dir, f"aldi_{args.variant}_5seeds_prq3.csv"), index=False)

    mean_user_ndcg20 = np.mean(np.stack(user_ndcg20_seeds, axis=0), axis=0)
    assert mean_user_ndcg20.shape == (3340,)
    np.save(os.path.join(adapter.output_dir, f"user_ndcg20_aldi_{args.variant}.npy"), mean_user_ndcg20)
    print(f"[HOÀN TẤT] Đã lưu user_ndcg20_aldi_{args.variant}.npy và các bảng CSV tại {out_dir}\n")

Writing run_aldi.py


In [6]:
# Chạy biến thể văn bản gốc (384 chiều) - Bắt buộc cho Bảng chính P-RQ1 và Bảng P-RQ2
!CUDA_VISIBLE_DEVICES=0 python run_aldi.py --variant raw --device cuda:0

=== KIỂM TRA MD5 VÀ BẢNG ÁNH XẠ CHỈ SỐ CHUNG ===
MD5 g_u.npy: 7a170558642de73d301970e8ce7f0a86 (Kỳ vọng: 7a170558...)
MD5 g_i.npy: d17ad9dfc4f4d6cb6de915c8649070d7 (Kỳ vọng: d17ad9df...)
MD5 raw_emb: f80445649ef8a8217d048bd3e7ac0ad5 | MD5 llm_emb: fc1819307be227a1544e1be0a4017393
MD5 llm_3d_emb: 1af436095463ad0af3ef1203ec0ebd6c
=== HOÀN TẤT KIỂM TRA: MỌI ASSERT ĐỀU ĐẠT CHUẨN! ===

=== [ALDI - raw] BẮT ĐẦU TINH CHỈNH 20 CẤU HÌNH (SEED 42) ===
Trial 01/20 | cfg={'lr': 0.0005, 'alpha': 0.1, 'beta': 0.1, 'tau': 1.0, 'wd': 1e-05} | Best Val NDCG@20=0.0905 (ep 75)
Trial 02/20 | cfg={'lr': 0.0001, 'alpha': 0.1, 'beta': 1.0, 'tau': 0.5, 'wd': 1e-05} | Best Val NDCG@20=0.0864 (ep 140)
Trial 03/20 | cfg={'lr': 0.001, 'alpha': 0.5, 'beta': 1.0, 'tau': 0.5, 'wd': 0.001} | Best Val NDCG@20=0.0860 (ep 15)
Trial 04/20 | cfg={'lr': 0.001, 'alpha': 0.1, 'beta': 1.0, 'tau': 1.0, 'wd': 0.001} | Best Val NDCG@20=0.0835 (ep 15)
Trial 05/20 | cfg={'lr': 0.001, 'alpha': 0.1, 'beta': 0.5, 'tau': 0.5, 'wd': 1e

In [7]:
# Chạy biến thể hồ sơ sư phạm LLM (384 chiều) - Phục vụ Bảng P-RQ2 đa mô hình
!CUDA_VISIBLE_DEVICES=0 python run_aldi.py --variant llm --device cuda:0

=== KIỂM TRA MD5 VÀ BẢNG ÁNH XẠ CHỈ SỐ CHUNG ===
MD5 g_u.npy: 7a170558642de73d301970e8ce7f0a86 (Kỳ vọng: 7a170558...)
MD5 g_i.npy: d17ad9dfc4f4d6cb6de915c8649070d7 (Kỳ vọng: d17ad9df...)
MD5 raw_emb: f80445649ef8a8217d048bd3e7ac0ad5 | MD5 llm_emb: fc1819307be227a1544e1be0a4017393
MD5 llm_3d_emb: 1af436095463ad0af3ef1203ec0ebd6c
=== HOÀN TẤT KIỂM TRA: MỌI ASSERT ĐỀU ĐẠT CHUẨN! ===

=== [ALDI - llm] BẮT ĐẦU TINH CHỈNH 20 CẤU HÌNH (SEED 42) ===
Trial 01/20 | cfg={'lr': 0.0005, 'alpha': 0.1, 'beta': 0.1, 'tau': 1.0, 'wd': 1e-05} | Best Val NDCG@20=0.0807 (ep 25)
Trial 02/20 | cfg={'lr': 0.0001, 'alpha': 0.1, 'beta': 1.0, 'tau': 0.5, 'wd': 1e-05} | Best Val NDCG@20=0.0846 (ep 80)
Trial 03/20 | cfg={'lr': 0.001, 'alpha': 0.5, 'beta': 1.0, 'tau': 0.5, 'wd': 0.001} | Best Val NDCG@20=0.0818 (ep 5)
Trial 04/20 | cfg={'lr': 0.001, 'alpha': 0.1, 'beta': 1.0, 'tau': 1.0, 'wd': 0.001} | Best Val NDCG@20=0.0808 (ep 5)
Trial 05/20 | cfg={'lr': 0.001, 'alpha': 0.1, 'beta': 0.5, 'tau': 0.5, 'wd': 1e-05

In [8]:
# Chạy biến thể ghép gốc + LLM (768 chiều) - Phục vụ Bảng P-RQ2 đa mô hình
!CUDA_VISIBLE_DEVICES=0 python run_aldi.py --variant fused --device cuda:0

=== KIỂM TRA MD5 VÀ BẢNG ÁNH XẠ CHỈ SỐ CHUNG ===
MD5 g_u.npy: 7a170558642de73d301970e8ce7f0a86 (Kỳ vọng: 7a170558...)
MD5 g_i.npy: d17ad9dfc4f4d6cb6de915c8649070d7 (Kỳ vọng: d17ad9df...)
MD5 raw_emb: f80445649ef8a8217d048bd3e7ac0ad5 | MD5 llm_emb: fc1819307be227a1544e1be0a4017393
MD5 llm_3d_emb: 1af436095463ad0af3ef1203ec0ebd6c
=== HOÀN TẤT KIỂM TRA: MỌI ASSERT ĐỀU ĐẠT CHUẨN! ===

=== [ALDI - fused] BẮT ĐẦU TINH CHỈNH 20 CẤU HÌNH (SEED 42) ===
Trial 01/20 | cfg={'lr': 0.0005, 'alpha': 0.1, 'beta': 0.1, 'tau': 1.0, 'wd': 1e-05} | Best Val NDCG@20=0.0810 (ep 5)
Trial 02/20 | cfg={'lr': 0.0001, 'alpha': 0.1, 'beta': 1.0, 'tau': 0.5, 'wd': 1e-05} | Best Val NDCG@20=0.0857 (ep 15)
Trial 03/20 | cfg={'lr': 0.001, 'alpha': 0.5, 'beta': 1.0, 'tau': 0.5, 'wd': 0.001} | Best Val NDCG@20=0.0753 (ep 25)
Trial 04/20 | cfg={'lr': 0.001, 'alpha': 0.1, 'beta': 1.0, 'tau': 1.0, 'wd': 0.001} | Best Val NDCG@20=0.0733 (ep 10)
Trial 05/20 | cfg={'lr': 0.001, 'alpha': 0.1, 'beta': 0.5, 'tau': 0.5, 'wd': 1e

In [9]:
# %%writefile run_cgrc_usim.py
# import argparse
# import os
# import numpy as np
# import torch
# from coldrec_adapter import ColdRecDataAdapter, evaluate_coldrec_embeddings
# from evaluator import evaluate

# parser = argparse.ArgumentParser()
# parser.add_argument("--model", type=str, default="usim", choices=["cgrc", "usim"])
# parser.add_argument("--variant", type=str, default="raw", choices=["raw", "llm", "fused"])
# parser.add_argument("--device", type=str, default="cuda:0")
# args = parser.parse_args()

# adapter = ColdRecDataAdapter()
# ds_dir = os.path.join("/kaggle/working/ColdRec_Data", f"MOOCCubeX_{args.variant}")
# out_dir = os.path.join(f"/kaggle/working/results_{args.model}", args.variant)
# os.makedirs(out_dir, exist_ok=True)

# # Quy trình thực hiện bên trong giống hệt ALDI:
# # 1. Tinh chỉnh 20 cấu hình (random.Random(42)) trên cold-val (cand_val_a, mask=hist_tr)
# # 2. Chạy cấu hình tốt nhất trên 5 seed [42, 2023, 2024, 1007, 999]
# # 3. Xuất u_emb [98220, 64] và i_emb [3769, 64] của mỗi seed đưa vào evaluate_coldrec_embeddings(...)
# # 4. Lưu kết quả tập (a), tập (b), P-RQ3 (.csv) và user_ndcg20_{args.model}_{args.variant}.npy

In [10]:
# # Chạy bắt buộc với đầu vào văn bản gốc (raw - 384 chiều)
# !CUDA_VISIBLE_DEVICES=0 python run_cgrc_usim.py --model usim --variant raw --device cuda:0

In [11]:
# !CUDA_VISIBLE_DEVICES=0 python run_cgrc_usim.py --model usim --variant llm --device cuda:0

In [12]:
# !CUDA_VISIBLE_DEVICES=0 python run_cgrc_usim.py --model usim --variant fused --device cuda:0

In [13]:
# Nén toàn bộ dữ liệu ColdRec_Data, thư mục kết quả và các tệp .npy (nếu đã sinh ra)
!zip -r -q /kaggle/working/all_coldrec_outputs.zip /kaggle/working/ColdRec_Data /kaggle/working/results_* $(ls /kaggle/working/user_ndcg20_*.npy 2>/dev/null)
!ls -lh /kaggle/working/

total 269M
drwxr-xr-x  7 root root 4.0K Oct 22 20:05 ALDI
-rw-r--r--  1 root root 268M Oct 22 20:41 all_coldrec_outputs.zip
drwxr-xr-x 10 root root 4.0K Oct 22 20:05 ColdRec
-rw-r--r--  1 root root  19K Oct 22 20:05 coldrec_adapter.py
drwxr-xr-x  5 root root 4.0K Oct 22 20:05 ColdRec_Data
-rw-r--r--  1 root root 114K Oct 22 20:41 __notebook__.ipynb
drwxr-xr-x  2 root root 4.0K Oct 22 20:05 __pycache__
drwxr-xr-x  6 root root 4.0K Oct 22 20:32 results_aldi
-rw-r--r--  1 root root 8.7K Oct 22 20:05 run_aldi.py
-rw-r--r--  1 root root  27K Oct 22 20:41 user_ndcg20_aldi_fused.npy
-rw-r--r--  1 root root  27K Oct 22 20:32 user_ndcg20_aldi_llm.npy
-rw-r--r--  1 root root  27K Oct 22 20:20 user_ndcg20_aldi_raw.npy
drwxr-xr-x  8 root root 4.0K Oct 22 20:05 USIM
